# B1: Permit Lifecycle Tracking

---

## Overview

Track the full permit lifecycle from zoning to occupancy.

**Pipeline Stages:**
1. Zoning Permit (Planning approval)
2. Building Permit (Construction authorization)
3. Inspections (Construction progress)
4. Certificate of Occupancy (Completion)

**Outputs:**
- `project_timelines.csv`
- Timeline metrics (days between stages)

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# Add modules to path
sys.path.insert(0, str(Path.cwd().parent))

# Import our modules
from modules.timeline_calculator import (
    calculate_days_between,
    get_project_timeline,
    classify_project_status,
    STATUS_ORDER
)
from modules.data_loader import load_csv, load_database

# Configuration
with open('../00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

DATA_DIR = Path(CONFIG['paths']['data_dir'])
DB_PATH = CONFIG['paths']['database']

print(f"Data directory: {DATA_DIR}")
print(f"Database: {DB_PATH}")

## 2. Load Project Data

In [ ]:
# Load housing projects
housing_path = Path(CONFIG['paths']['housing_projects'])
df_projects = load_csv(housing_path)

if df_projects is not None:
    print(f"Loaded {len(df_projects)} projects")
    print(f"\nColumns: {df_projects.columns.tolist()}")
    
    # Show sample
    display(df_projects[['address_display', 'permits', 'status', 'year']].head(10))

## 3. Parse Permit Information

Extract individual permits from the permits column.

In [ ]:
def parse_permits(permits_str):
    """
    Parse comma-separated permits into list.
    """
    if pd.isna(permits_str):
        return []
    return [p.strip() for p in str(permits_str).split(',')]

def identify_permit_type(permit_id):
    """
    Identify permit type from ID.
    """
    permit_id = str(permit_id).upper()
    
    if permit_id.startswith('ZP'):
        return 'Zoning Permit'
    elif permit_id.startswith('PLN'):
        return 'Planning Record'
    elif permit_id.startswith('BP') or permit_id.startswith('B'):
        return 'Building Permit'
    elif permit_id.startswith('CO'):
        return 'Certificate of Occupancy'
    else:
        return 'Unknown'

# Test
if df_projects is not None:
    sample_permits = df_projects['permits'].dropna().iloc[0]
    print(f"Sample permits string: {sample_permits}")
    print(f"Parsed: {parse_permits(sample_permits)}")

In [ ]:
# Analyze all permits
if df_projects is not None:
    all_permits = []
    
    for idx, row in df_projects.iterrows():
        permits = parse_permits(row['permits'])
        for p in permits:
            all_permits.append({
                'project_id': idx,
                'address': row.get('address_display', ''),
                'permit_id': p,
                'permit_type': identify_permit_type(p)
            })
    
    df_permits = pd.DataFrame(all_permits)
    
    print(f"Total permits: {len(df_permits)}")
    print(f"\nPermit types:")
    print(df_permits['permit_type'].value_counts())

## 4. Link Permits by Address and APN

Connect related permits to track full project lifecycle.

In [ ]:
# Group permits by project
if 'df_permits' in dir():
    permit_summary = df_permits.groupby('project_id').agg({
        'permit_id': lambda x: ', '.join(x),
        'permit_type': lambda x: ', '.join(x.unique())
    }).reset_index()
    
    permit_summary.columns = ['project_id', 'all_permits', 'permit_types']
    
    print(f"Projects with permits: {len(permit_summary)}")
    display(permit_summary.head(10))

## 5. Calculate Timeline Metrics

Calculate days between permit stages.

In [ ]:
# Add timeline metrics to projects
if df_projects is not None:
    df_timelines = df_projects.copy()
    
    # Count permits per project
    df_timelines['permit_count'] = df_timelines['permits'].apply(
        lambda x: len(parse_permits(x))
    )
    
    # Classify current status
    if 'status' in df_timelines.columns:
        df_timelines['status_category'] = df_timelines['status'].apply(
            classify_project_status
        )
    
    print("Timeline Summary:")
    print(f"  Total projects: {len(df_timelines)}")
    print(f"  Avg permits/project: {df_timelines['permit_count'].mean():.1f}")
    
    if 'status_category' in df_timelines.columns:
        print(f"\nProjects by status category:")
        print(df_timelines['status_category'].value_counts())

## 6. Pipeline Stage Analysis

In [ ]:
# Display pipeline stages
print("Housing Development Pipeline Stages:")
print("="*50)

for i, stage in enumerate(STATUS_ORDER, 1):
    count = len(df_timelines[df_timelines['status_category'] == stage]) if 'df_timelines' in dir() else 0
    print(f"{i}. {stage.upper():20} - {count} projects")

## 7. Export Timeline Data

In [ ]:
# Export timeline data
if 'df_timelines' in dir():
    output_path = DATA_DIR / 'project_timelines.csv'
    df_timelines.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

---

## Summary

This notebook:
- Parsed permit IDs and identified types
- Linked permits by address/APN
- Calculated timeline metrics
- Classified projects by pipeline stage

**Next:** Run `B2_status_classification.ipynb` for detailed status tracking.